In [61]:
# Imports
import pandas as pd
import numpy as np
import os
from functools import reduce


In [62]:
# Importing DB...
BASE_DB = "mimic-iv-clinical-database-demo-2.2"
HOSP_PATH = os.path.join(BASE_DB, "hosp")
ICU_PATH = os.path.join(BASE_DB, "icu")

# Loading Tables...
patients = pd.read_csv(f'{HOSP_PATH}/patients.csv.gz')
admissions = pd.read_csv(f'{HOSP_PATH}/admissions.csv.gz')
lab_events = pd.read_csv(f'{HOSP_PATH}/labevents.csv.gz')

chart_events = pd.read_csv(f'{ICU_PATH}/chartevents.csv.gz')
icu_stays = pd.read_csv(f'{ICU_PATH}/icustays.csv.gz')
d_items = pd.read_csv(f'{ICU_PATH}/d_items.csv.gz') 



# Step 0 : Data Exploring

In [63]:
######## Charevents table: understanding clinical variables
# each itemid represents the SAME clinical measurements in the ICU system (heart rate, blood pressure, respiratory rate, oxygen saturation)
# For example: the heart rate measurements could be mapped to multiple item_ids (for the same patient & icu stay)
df_vitals = chart_events[
    (chart_events["valueuom"].str.lower() == "bpm") &
    (chart_events["stay_id"] == 32604416)
]
df_unique_itemids = df_vitals.drop_duplicates(subset=["itemid"])
print(df_unique_itemids[["subject_id","stay_id","itemid","value" ,"valuenum", "valueuom"]].head())


    subject_id   stay_id  itemid value  valuenum valueuom
5     10005817  32604416  224751    52      52.0      bpm
11    10005817  32604416  220047    55      55.0      bpm
36    10005817  32604416  220046   120     120.0      bpm
69    10005817  32604416  220045    80      80.0      bpm


# Step 1: Data Extraction | Feature Definition

In [64]:
#convert to datetime
icu_stays['intime'] = pd.to_datetime(icu_stays['intime'])
icu_stays['outtime'] = pd.to_datetime(icu_stays['outtime'])

# Select first ICU stay per patient
icu_stays = icu_stays.sort_values(by="intime")
icu_stay_first = icu_stays.groupby("subject_id").first()

# add the patients' demographics based on the chosen icu stay
df = icu_stay_first.merge(patients, on="subject_id")

# add relevant hospital admission
df = df.merge(admissions, on=["subject_id", "hadm_id"])

# Keep relevant columns
df = df[[
     "subject_id","hadm_id", 
     "stay_id", "first_careunit",
     "anchor_age", "gender",
     "admission_type", "hospital_expire_flag",
     "intime", "outtime"
]]
df.head()



,subject_id,hadm_id,stay_id,first_careunit,anchor_age,gender,admission_type,hospital_expire_flag,intime,outtime
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),52,F,EW EMER.,0,2180-07-23 14:00:00,2180-07-23 23:50:47
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),55,F,EW EMER.,0,2157-11-20 19:18:02,2157-11-21 22:08:00
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),46,F,EW EMER.,0,2110-04-11 15:52:22,2110-04-12 23:59:56
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),80,F,EW EMER.,0,2156-04-12 16:24:18,2156-04-17 15:57:08
4,10002495,24982426,36753294,Coronary Care Unit (CCU),81,M,URGENT,0,2141-05-22 20:18:01,2141-05-27 22:24:02


In [65]:
####### Define chartevents for 24H window (var:24H_endtime)
# 1- Calculate the 24H_endtime based on each icu_stay intime
# 2- Extract the chartevents done in those 24H
df["24H_endtime"] = df["intime"] + pd.Timedelta(hours=24)
chart_events["charttime"] = pd.to_datetime(chart_events["charttime"])
chart_events = chart_events.merge(
    df[["stay_id", "intime", "24H_endtime"]],
    on="stay_id"
)

chart_events_24H = chart_events[
    (chart_events["charttime"] >= chart_events["intime"]) &
    (chart_events["charttime"] <= chart_events["24H_endtime"])
]

chart_events_24H.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,intime,24H_endtime
0,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:45:00,225054,On,NaN,NaN,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01
1,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:43:00,223769,100,100.0,%,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01
2,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:47:00,223956,Atrial demand,NaN,NaN,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01
3,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:47:00,224866,Yes,NaN,NaN,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01
4,10005817,20626031,32604416,6770.0,2132-12-16,2132-12-15 23:45:00,227341,No,0.0,NaN,0.0,2132-12-15 09:29:01,2132-12-16 09:29:01


In [66]:
#Different vital signs extracted in the 24H window
chart_events_24H["valueuom"].unique()

<StringArray>
[                nan,                 '%',               'bpm',
              'mmHg',                'mV',          'insp/min',
                'mA',                '°C',                'kg',
             'L/min',               'sec',             'cmH2O',
                'mL',                'cm',              'g/dl',
             'mEq/L',             'units',             'mg/dL',
              'K/uL',            'mmol/L',              'Inch',
                '°F',              'IU/L',             'ng/mL',
              'g/dL',            'ml/min',             'ug/mL',
               'min',              'mg/L',             'mm/hr',
           'mOsm/kg',             'ug/dL',             'pg/mL',
             'ml/hr',            'umol/L',             'mL/m2',
          'L/min/m2',             'ml/kg', 'dynes.sec.cm-5/m2']
Length: 39, dtype: str

In [67]:
# 3- Map each vital sign variable (heart rate, blood pressure, respiratory rate, oxygen saturation) to the corresponding itemids
def vitals_mapping(items_tb,column,vital_sign):
    return items_tb[items_tb[column].str.contains(vital_sign, case=False)]["itemid"].tolist()

vital_sign_ids = {
    "HR": vitals_mapping(d_items, "label", "heart rate"), # Heart Rate
    "BP": vitals_mapping(d_items, "label", "blood pressure"), # Blood Pressure
    "RR": vitals_mapping(d_items, "label", "respiratory rate"), # Respiratory Rate
    "SpO2": vitals_mapping(d_items, "label", "spo2"), # O2 Saturation
    "TemperatureF": vitals_mapping(d_items, "label", "Temperature Fahrenheit") # TemperatureF
    #"Creatinine": vitals_mapping(d_items, "label", "creatinine"), # Creatinine
    #"Magnesium": vitals_mapping(d_items, "label", "magnesium") # Magnesium
   
}

print(vital_sign_ids)

{'HR': [220047, 220046, 220045], 'BP': [227539, 220058, 220056, 223752, 227538, 227537, 223751, 220052, 227242, 220180, 227243, 220051, 220181, 224643, 220179, 220050, 224167], 'RR': [224688, 224690, 220210, 224689], 'SpO2': [226253, 229862], 'TemperatureF': [223761]}


In [ ]:
# 4- Add the vital signs values in the 24H window to the main df based on the itemids mapping

# for vital_sign, itemids in vital_sign_ids.items():
#     df_vital = chart_events_24H[chart_events_24H["itemid"].isin(itemids)]
#     df_vital = df_vital[["stay_id", "valuenum"]].groupby("stay_id").median()
#     df_vital.rename(columns={"valuenum": vital_sign}, inplace=True)
#     df = df.merge(df_vital, on="stay_id", how="left")   

##### ICU data is all about extremes and critical events (max, min)  
# the median will help up determine the typical state
vitals_list = []
for vital, ids in vital_sign_ids.items():
    temp = chart_events_24H[chart_events_24H["itemid"].isin(ids)]
    
    agg = temp.groupby("stay_id")["valuenum"].agg(
        ["median", "min", "max"]
    ).reset_index()
    
    agg.columns = ["stay_id",
                   f"{vital}_median",
                   f"{vital}_min",
                   f"{vital}_max"]
    
    vitals_list.append(agg)

# Merge all vitals in a DataFrame
vitals_df = reduce(lambda left, right: pd.merge(left, right, on="stay_id", how="outer"), vitals_list)

vitals_df.head()


,stay_id,HR_median,HR_min,HR_max,BP_median,BP_min,BP_max,RR_median,RR_min,RR_max,SpO2_median,SpO2_min,SpO2_max,TemperatureF_median,TemperatureF_min,TemperatureF_max
0,30057454,109.0,60.0,130.0,75.5,47.0,140.0,18.0,13.0,25.0,85.0,85.0,85.0,97.90,97.7,98.7
1,30101877,93.0,50.0,123.0,89.5,49.0,166.0,20.0,4.0,25.0,85.0,85.0,85.0,100.20,99.3,100.5
2,30458995,85.0,50.0,130.0,88.0,-8.0,170.0,17.0,14.0,24.0,88.0,88.0,88.0,98.00,97.4,98.3
3,30585761,71.5,60.0,100.0,74.5,35.0,150.0,20.0,10.0,24.0,85.0,85.0,85.0,98.05,97.4,98.2
4,30665396,90.0,50.0,120.0,76.0,51.0,150.0,19.0,0.0,36.0,85.0,85.0,85.0,98.20,97.9,98.9


In [69]:
df = df.merge(vitals_df, on="stay_id", how="left")
df.head()

,subject_id,hadm_id,stay_id,first_careunit,anchor_age,gender,admission_type,hospital_expire_flag,intime,outtime,...,BP_max,RR_mean,RR_min,RR_max,SpO2_mean,SpO2_min,SpO2_max,TemperatureF_mean,TemperatureF_min,TemperatureF_max
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),52,F,EW EMER.,0,2180-07-23 14:00:00,2180-07-23 23:50:47,...,160.0,20.700000,16.0,24.0,86.500000,85.0,88.0,98.966667,98.7,99.5
1,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),55,F,EW EMER.,0,2157-11-20 19:18:02,2157-11-21 22:08:00,...,160.0,21.320000,13.0,27.0,88.000000,88.0,88.0,99.062500,98.1,100.8
2,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),46,F,EW EMER.,0,2110-04-11 15:52:22,2110-04-12 23:59:56,...,160.0,17.360000,13.0,21.0,85.000000,85.0,85.0,97.800000,97.5,98.3
3,10002428,28662225,33987268,Medical Intensive Care Unit (MICU),80,F,EW EMER.,0,2156-04-12 16:24:18,2156-04-17 15:57:08,...,160.0,24.448980,18.0,34.0,86.428571,85.0,89.0,100.300000,98.6,102.9
4,10002495,24982426,36753294,Coronary Care Unit (CCU),81,M,URGENT,0,2141-05-22 20:18:01,2141-05-27 22:24:02,...,160.0,21.710526,16.0,28.0,85.000000,85.0,85.0,98.350000,98.0,98.7
